# Ablation: `web_search` availability vs. eval rollup

[Issue #14](https://github.com/davidzornek/adk/issues/14) — a comparative eval write-up, not new
library code. It reuses the hand-written eval cases and metrics from
[`eval_plan_then_act_demo.ipynb`](../demos/eval_plan_then_act_demo.ipynb) and runs them through
`adk.demos.plan_then_act_demo.DemoPlanThenActAgent` twice:

1. **baseline** — `web_search` and `calculate` both live, as normal.
2. **ablated** — `web_search` always fails; `calculate` untouched.

The question this is evidence for: does the eval harness's `task_success` /
`plan_execution_alignment` split actually distinguish *"the plan was right but a tool failed"*
from *"the plan was wrong"*? An ablation that reliably breaks one tool and leaves the planner's
routing logic untouched is a clean way to check that, and it doubles as a small case study in
`DegradedModeExecutor`'s fault boundary.

**Directional / small-N**: 3 hand-written cases, one run per case per configuration, no repeats.
This is not a statistically rigorous claim — see the closing section for what it can and can't
support.

## Setup

Same two keys as the other demo notebooks:

- `ANTHROPIC_API_KEY` — https://console.anthropic.com/
- `TAVILY_API_KEY` — https://tavily.com/

Copy `.env.example` (repo root) to `.env` and paste your keys in there — `load_dotenv()` below
loads it into this process.

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

for key in ("ANTHROPIC_API_KEY", "TAVILY_API_KEY"):
    print(f"{key}: {'set' if os.environ.get(key) else 'MISSING'}")


ANTHROPIC_API_KEY: set
TAVILY_API_KEY: set


## The cases

The same three hand-written cases as `eval_plan_then_act_demo.ipynb` (`docs/demos/`), so the
baseline rollup below is directly comparable to that notebook's:

- **`population_lookup`** — pure lookup, should route to `search` only. Search-dependent.
- **`arithmetic`** — pure arithmetic, no lookup needed, should route to `calc` only.
  Search-*independent* — the control case: ablating `web_search` should not touch this one.
- **`combined_population`** — two `search` steps (France, Germany) then one `calc` step. Note
  that in this pattern `calc`'s `tool_args` are fixed by the planner *before* any tool runs (see
  `plan_then_act_demo`'s module docstring), so the calc step never actually consumes the search
  results — it degrades or succeeds independently of them.

In [2]:
from adk.eval_harness.cases import EvalCase, ExpectedStep
from adk.eval_harness.local_harness import rollup, run_and_score
from adk.eval_harness.metrics import (
    latency,
    plan_execution_alignment,
    steps_to_completion,
    task_success,
    token_counts,
)

CASES = [
    EvalCase(
        id="population_lookup",
        task="What is the current population of France?",
        expected_steps=[ExpectedStep(executor_id="search", tool_name="web_search")],
    ),
    EvalCase(
        id="arithmetic",
        task="What is 482 times 17?",
        expected_steps=[ExpectedStep(executor_id="calc", tool_name="calculate")],
    ),
    EvalCase(
        id="combined_population",
        task=(
            "Look up the current population of France and Germany, then calculate their "
            "combined population."
        ),
        expected_steps=[
            ExpectedStep(executor_id="search", tool_name="web_search"),
            ExpectedStep(executor_id="search", tool_name="web_search"),
            ExpectedStep(executor_id="calc", tool_name="calculate"),
        ],
    ),
]

METRICS = [task_success, steps_to_completion, plan_execution_alignment, latency, token_counts]


## Two agent configurations

`DemoPlanThenActAgent` (baseline) is imported unmodified from `adk.demos.plan_then_act_demo`.

The ablated variant, `AblatedPlanThenActAgent`, is assembled from the exact same building
blocks `DemoPlanThenActAgent._build_graph` uses (`build_plan_then_act_planner`,
`AnthropicRunnable`, `DegradedModeExecutor`, `build_plan_then_act_graph`) — same planner system
prompt, same config, same `calculate` tool. The *only* change is what the `search` executor's
`web_search` tool does: instead of a live Tavily call, it always raises. The planner still knows
about and plans for `web_search` exactly as before; `DegradedModeExecutor`'s own fault boundary
is what turns that into a `"degraded"` step, the same way a real Tavily outage would. This is
what makes it an ablation of the *tool*, not a change to the *planner's routing logic* — routing
should be unaffected, and only `task_success` on search-dependent cases should move.

In [3]:
from typing import Any

from anthropic import Anthropic

from adk.anthropic_client import get_anthropic_client
from adk.anthropic_runnables import AnthropicRunnable
from adk.demos.plan_then_act_demo import (
    DEFAULT_MODEL,
    DRAFTER_SYSTEM_PROMPT,
    PLANNER_SYSTEM_PROMPT,
    DemoPlanThenActAgent,
    calculate,
    default_config,
)
from adk.planner_executor.base import PlannerExecutorBase
from adk.planner_executor.executor import DegradedModeExecutor
from adk.planner_executor.graph import build_plan_then_act_graph
from adk.planner_executor.planner import build_plan_then_act_planner


def disabled_web_search(args: dict[str, Any], ctx: dict[str, Any]) -> dict[str, Any]:
    """Ablation stand-in for ``web_search``: always fails, as if the provider were down."""
    raise RuntimeError("web_search is disabled for this ablation run")


class AblatedPlanThenActAgent(PlannerExecutorBase):
    """``DemoPlanThenActAgent`` with ``web_search`` swapped for a tool that always fails."""

    def __init__(self, *, client: Anthropic | None = None, model: str = DEFAULT_MODEL) -> None:
        self._client = client or get_anthropic_client()
        self._model = model
        super().__init__(config=default_config())

    @property
    def variant(self) -> str:
        return "plan_then_act_demo_ablated_no_search"

    def _input_to_state(self, input: dict[str, Any]) -> dict[str, Any]:
        return {"task": input["task"], "context": input.get("context", {})}

    def _build_graph(self) -> Any:
        planner = build_plan_then_act_planner(
            self._client, model=self._model, system_prompt=PLANNER_SYSTEM_PROMPT,
        )
        drafter = AnthropicRunnable(
            self._client, model=self._model, system_prompt=DRAFTER_SYSTEM_PROMPT,
        )
        executors = {
            "search": DegradedModeExecutor(
                executor_id="search",
                tools={"web_search": disabled_web_search},
                degraded_mode="Web search is temporarily unavailable.",
            ),
            "calc": DegradedModeExecutor(
                executor_id="calc",
                tools={"calculate": calculate},
                degraded_mode="Calculation failed.",
            ),
        }
        return build_plan_then_act_graph(
            planner=planner,
            executors=executors,
            drafter=drafter,
            drafter_system=DRAFTER_SYSTEM_PROMPT,
            validator=None,
            config=self.config,
        )


baseline_agent = DemoPlanThenActAgent()
ablated_agent = AblatedPlanThenActAgent()


## Running both configurations

Six live Anthropic + Tavily round trips total (three cases x two configurations) — this cell
takes a little while. `TAVILY_API_KEY` is still required even for the ablated run: the search
executor's tool is swapped for `disabled_web_search`, but Tavily itself is never called in that
configuration — the key is only exercised by the baseline run.

In [4]:
baseline_rows = run_and_score(CASES, baseline_agent, METRICS)
ablated_rows = run_and_score(CASES, ablated_agent, METRICS)


executor search degraded: web_search is disabled for this ablation run


executor search degraded: web_search is disabled for this ablation run


executor search degraded: web_search is disabled for this ablation run


## Scored rows, side by side

In [5]:
def print_rows(label: str, rows: list[dict]) -> None:
    print(f"--- {label} ---")
    for row in rows:
        print(f"{row['case_id']!r}:")
        for key, value in row.items():
            if key == "case_id":
                continue
            print(f"  {key}: {value}")


print_rows("baseline", baseline_rows)
print()
print_rows("ablated", ablated_rows)


--- baseline ---
'population_lookup':
  run_id: e83cf8ed-1784-4db9-bf06-f73d98c07183
  success: True
  n_steps: 1
  n_degraded_steps: 0
  degraded_step_indices: []
  steps_to_completion: 1
  aligned: True
  n_expected_steps: 1
  n_executed_steps: 1
  mismatches: []
  latency_ms: 5525.602750014514
  tokens_in: 3308
  tokens_out: 240
  tokens_total: 3548
'arithmetic':
  run_id: 1bc99360-397b-4267-85ca-14614a73ecb8
  success: True
  n_steps: 1
  n_degraded_steps: 0
  degraded_step_indices: []
  steps_to_completion: 1
  aligned: True
  n_expected_steps: 1
  n_executed_steps: 1
  mismatches: []
  latency_ms: 2643.7571249553002
  tokens_in: 1434
  tokens_out: 88
  tokens_total: 1522
'combined_population':
  run_id: 76f6272e-2167-4070-be75-1b949df1481a
  success: True
  n_steps: 3
  n_degraded_steps: 0
  degraded_step_indices: []
  steps_to_completion: 3
  aligned: True
  n_expected_steps: 3
  n_executed_steps: 3
  mismatches: []
  latency_ms: 4833.251625008415
  tokens_in: 4125
  tokens_out:

## Rollups and the diff

`rollup()` is the same function used in `eval_plan_then_act_demo.ipynb`. The diff below is the
"small script" the issue calls for — no new library code, just a dict subtraction over whatever
keys both rollups share.

In [6]:
baseline_rollup = rollup(baseline_rows)
ablated_rollup = rollup(ablated_rows)

print("baseline:", baseline_rollup)
print("ablated: ", ablated_rollup)


def diff_rollups(before: dict, after: dict) -> dict:
    """Per-key delta (after - before) over keys present in both rollups."""
    return {
        key: after[key] - before[key]
        for key in before
        if key in after and isinstance(before[key], (int, float))
    }


diff_rollups(baseline_rollup, ablated_rollup)


baseline: {'n_cases': 3, 'pass_rate': 1.0, 'avg_steps_to_completion': 1.6666666666666667, 'alignment_rate': 1.0}
ablated:  {'n_cases': 3, 'pass_rate': 0.3333333333333333, 'avg_steps_to_completion': 1.6666666666666667, 'alignment_rate': 1.0}


{'n_cases': 0,
 'pass_rate': -0.6666666666666667,
 'avg_steps_to_completion': 0.0,
 'alignment_rate': 0.0}

## Per-case read: which cases moved, and on which metric

`task_success` (did the run finish with zero degraded steps) is expected to flip on the two
search-dependent cases and stay put on `arithmetic`. `plan_execution_alignment` (did the planner
route to the right executor/tool, independent of whether the tool call itself succeeded) is
expected to stay aligned on all three — the planner never sees the ablation, only
`DegradedModeExecutor` does.

In [7]:
by_case = {
    "baseline": {r["case_id"]: r for r in baseline_rows},
    "ablated": {r["case_id"]: r for r in ablated_rows},
}

for case in CASES:
    b = by_case["baseline"][case.id]
    a = by_case["ablated"][case.id]
    print(
        f"{case.id:20s} success: {str(b['success']):5s} -> {str(a['success']):5s}   "
        f"aligned: {str(b['aligned']):5s} -> {str(a['aligned']):5s}   "
        f"degraded_steps: {b['n_degraded_steps']} -> {a['n_degraded_steps']}",
    )


population_lookup    success: True  -> False   aligned: True  -> True    degraded_steps: 0 -> 1
arithmetic           success: True  -> True    aligned: True  -> True    degraded_steps: 0 -> 0
combined_population  success: True  -> False   aligned: True  -> True    degraded_steps: 0 -> 2


## Honest read of the results

**Directional, small-N — 3 hand-written cases, single run each, no repeats or confidence
intervals. This is not a statistically rigorous claim about the agent's reliability under tool
outages; it's a worked example of what one such outage looks like through this harness.**

What the numbers above should show, and why:

- **`arithmetic` is the control.** It doesn't touch `web_search`, so both configurations should
  score it identically. If it moved too, that's a sign the ablation leaked somewhere it
  shouldn't have (e.g. the planner's prompt or the calc tool itself), not a real ablation effect.
- **`population_lookup` and `combined_population` are the treatment.** Both depend on
  `web_search`; both should show `task_success` flip to `False` in the ablated run, surfaced via
  `n_degraded_steps > 0` — `DegradedModeExecutor` catching the forced `RuntimeError` and turning
  it into a `"degraded"` step rather than crashing the run.
- **`plan_execution_alignment` is the metric that should *not* move.** The planner's system
  prompt and available `output_plan` schema are identical in both configurations — only the tool
  implementation the executor calls is different — so a correctly-routed step (`executor_id`
  and `tool` matching `expected_steps`) stays correctly routed even when its execution degrades.
  If alignment had dropped here too, that would suggest the planner itself was reacting to the
  outage (e.g. skipping the search step entirely, or substituting `calculate`), which is a
  materially different — and worse — failure mode than a clean tool degradation.
- **This demonstrates why the harness carries both metrics rather than one.** A single
  pass/fail number can't distinguish "the plan was right and a dependency failed" from "the plan
  was wrong" — those call for different fixes (retry/fallback infrastructure vs. prompt or
  routing changes). Seeing `success=False` alongside `aligned=True` on the ablated
  search-dependent cases is exactly that distinction made visible in one row.